# Overture motorways — interactive view

Fetches Overture `transportation` motorway segments (ramps dropped via the
`is_link` road flag) for the freeway-hex bbox using
`scripts/build_road_table.py`, loads them into a GeoDataFrame, and renders
with lonboard `viz`.

This is the road geometry that feeds the H3 columnar join into
`data/road_table.parquet`.

In [1]:
import sys

sys.path.append("scripts")

import geopandas as gpd
import pyarrow.parquet as pq
from shapely import wkb

from overturemaps.core import get_latest_release
from build_road_table import MOTORWAYS, fetch_motorways, hex_bbox

bbox = hex_bbox()
release = get_latest_release()
print("release", release, "bbox", bbox)

# refetch=True re-pulls from Overture. The cached parquet is stale (pre-ramp-
# filter), so set True on the first run after changing the WHERE clause.
fetch_motorways(bbox, release, refetch=True)

release 2026-05-20.0 bbox (-124.78560603210607, 25.02821309446701, -67.3632084036063, 49.57700082876189)
[motorways] querying Overture 2026-05-20.0 within (-124.78560603210607, 25.02821309446701, -67.3632084036063, 49.57700082876189) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[motorways] wrote data/overture_motorways.parquet (135196 rows)


In [2]:
df = pq.read_table(MOTORWAYS).to_pandas()
df["geometry"] = df.geometry_wkb.map(lambda b: wkb.loads(bytes(b)))
gdf = gpd.GeoDataFrame(
    df[["road_id", "road_name"]],
    geometry=df.geometry,
    crs="EPSG:4326",
)
print(len(gdf), "motorway segments |", gdf.road_name.notna().sum(), "named")
gdf.head()

135196 motorway segments | 85035 named


,road_id,road_name,geometry
0,6a1c2670-25ac-4a08-9192-7fefeb78924f,Autopista Culiacan - Los Mochis,"LINESTRING (-107.87704 25.07103, -107.87655 25..."
1,5de0b77c-4b5b-4ecf-a412-67b5ec0c91dc,Autopista Culiacan - Los Mochis,"LINESTRING (-107.86473 25.05958, -107.8659 25...."
2,24daf638-11db-4e88-ab2a-51e1c0d35caa,Autopista Culiacan - Los Mochis,"LINESTRING (-107.87984 25.07365, -107.87831 25..."
3,a585f39a-1300-4635-9445-63528c1df8a1,Autopista Culiacan - Los Mochis,"LINESTRING (-107.87686 25.07099, -107.87809 25..."
4,29b1f349-63d6-4c87-aa6b-1b97d453ce7a,Autopista Culiacan - Los Mochis,"LINESTRING (-107.86986 25.0644, -107.8733 25.0..."


In [3]:
from lonboard import viz

viz(gdf)